# 📋 입찰메이트 RAG 파이프라인 — 전체 작동 검증 노트북

**B안: OpenAI API 기반 서비스형 RAG** 의 4단계를 순서대로 실행하고 각 단계 성공/실패를 확인합니다.

| 단계 | 스크립트 | 주요 산출물 |
|---|---|---|
| 1. 환경 점검 | — | `.env`, 패키지, 모델명 |
| 2. 인덱싱 | `scripts/index_documents.py` | `data/processed/*_chunks.json`, `data/vectordb/` |
| 3. 앱 파이프라인 | `app.py` (RAGPipeline 직접 호출) | 답변 + 참고 문서 |
| 4. 평가 | `scripts/run_evaluation.py` | `evaluation/*.csv / *.json` |
| 5. 릴리즈 게이트 | `scripts/check_release_gate.py` | `evaluation/gate_report_core.json` |

> **셀을 위에서 아래로 순서대로 실행하세요.** 각 셀 하단의 `✅ PASS` / `❌ FAIL` 출력으로 진행 여부를 판단합니다.

---
## Step 0. 공통 유틸리티

In [1]:
import subprocess, sys, os, json
from pathlib import Path

# 프로젝트 루트를 Python 경로에 추가
PROJECT_ROOT = Path().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def check(label: str, condition: bool, detail: str = ""):
    """PASS/FAIL 출력 헬퍼"""
    icon = "✅" if condition else "❌"
    status = "PASS" if condition else "FAIL"
    print(f"{icon} [{status}] {label}")
    if detail:
        print(f"       {detail}")
    return condition

def run_script(cmd: list, label: str) -> bool:
    """subprocess로 스크립트 실행 후 결과 반환"""
    print(f"\n▶ 실행: {' '.join(cmd)}\n{'-'*60}")
    result = subprocess.run(cmd, capture_output=False, text=True)
    success = result.returncode == 0
    print(f"{'-'*60}")
    check(label, success, f"exit code: {result.returncode}")
    return success

print("✅ 유틸리티 로드 완료")
print(f"   프로젝트 루트: {PROJECT_ROOT}")

✅ 유틸리티 로드 완료
   프로젝트 루트: /home/jihoon/sprint_ai_07_middle


---
## Step 1. 환경 점검
패키지 설치, `.env` API 키, 파일 구조, 모델명을 사전에 확인합니다.

In [2]:
# 1-1. 핵심 패키지 import 가능 여부
print("=" * 60)
print("[1-1] 핵심 패키지 로드")
print("=" * 60)

packages = [
    ("openai",       "openai"),
    ("langchain",    "langchain"),
    ("chromadb",     "chromadb"),
    ("streamlit",    "streamlit"),
    ("dotenv",       "python-dotenv"),
    ("pandas",       "pandas"),
    ("pyarrow",      "pyarrow"),
    ("rouge_score",  "rouge-score"),
    ("nltk",         "nltk"),
]

all_ok = True
for mod, pkg in packages:
    try:
        __import__(mod)
        check(f"import {mod}", True)
    except ImportError:
        check(f"import {mod}", False, f"pip install {pkg}")
        all_ok = False

if not all_ok:
    print("\n⚠️  누락 패키지가 있습니다. 아래 셀을 실행해 설치하세요.")

[1-1] 핵심 패키지 로드
✅ [PASS] import openai
✅ [PASS] import langchain
✅ [PASS] import chromadb
✅ [PASS] import streamlit
✅ [PASS] import dotenv
✅ [PASS] import pandas
✅ [PASS] import pyarrow
✅ [PASS] import rouge_score
✅ [PASS] import nltk


In [3]:
# 1-2. (필요 시) 패키지 일괄 설치 — 위 셀에서 FAIL이 있을 때만 실행
# !pip install -r requirements.txt

In [4]:
# 1-3. .env 로드 및 API 키 확인
print("=" * 60)
print("[1-3] 환경변수 (.env) 확인")
print("=" * 60)

from dotenv import load_dotenv
load_dotenv()

env_file_ok  = check(".env 파일 존재",        Path(".env").exists())
api_key_ok   = check("OPENAI_API_KEY 설정됨",  bool(os.getenv("OPENAI_API_KEY")),
                     "없으면 .env에 OPENAI_API_KEY=sk-... 추가")

if api_key_ok:
    key = os.getenv("OPENAI_API_KEY")
    print(f"       키 앞 8자리: {key[:8]}...")

[1-3] 환경변수 (.env) 확인
✅ [PASS] .env 파일 존재
✅ [PASS] OPENAI_API_KEY 설정됨
       없으면 .env에 OPENAI_API_KEY=sk-... 추가
       키 앞 8자리: sk-proj-...


In [5]:
# 1-4. 프로젝트 파일 구조 확인
print("=" * 60)
print("[1-4] 필수 파일 존재 확인")
print("=" * 60)

required_files = [
    "app.py",
    "scripts/index_documents.py",
    "scripts/run_evaluation.py",
    "scripts/check_release_gate.py",
    "configs/config.py",
    "configs/paths.py",
    "src/rag_pipeline.py",
    "src/document_loader.py",
    "src/chunker.py",
    "src/embedder.py",
    "src/evaluator.py",
    "requirements.txt",
]

for f in required_files:
    check(f, Path(f).exists())

[1-4] 필수 파일 존재 확인
✅ [PASS] app.py
✅ [PASS] scripts/index_documents.py
✅ [PASS] scripts/run_evaluation.py
✅ [PASS] scripts/check_release_gate.py
✅ [PASS] configs/config.py
✅ [PASS] configs/paths.py
✅ [PASS] src/rag_pipeline.py
✅ [PASS] src/document_loader.py
✅ [PASS] src/chunker.py
✅ [PASS] src/embedder.py
✅ [PASS] src/evaluator.py
✅ [PASS] requirements.txt


---
## Step 2. 인덱싱
`scripts/index_documents.py` 를 실행해 청크 생성 및 ChromaDB 저장을 확인합니다.

In [6]:
# 2-0. 인덱싱 설정값 (필요 시 수정)
COLLECTION   = "rfp_chunk600"          # 사용할 ChromaDB 컬렉션 이름
PARQUET_PATH = "data/autorag_csv/corpus.parquet"  # corpus.parquet 경로

# parquet 파일 존재 여부 사전 확인
parquet_ok = check("corpus.parquet 존재", Path(PARQUET_PATH).exists(),
                   f"경로: {PARQUET_PATH}")
if not parquet_ok:
    print("\n⚠️  parquet 파일이 없으면 --from-parquet 옵션 없이 실행합니다.")
    print("   일반 문서 폴더(data/) 기반 인덱싱으로 전환됩니다.")

❌ [FAIL] corpus.parquet 존재
       경로: data/autorag_csv/corpus.parquet

⚠️  parquet 파일이 없으면 --from-parquet 옵션 없이 실행합니다.
   일반 문서 폴더(data/) 기반 인덱싱으로 전환됩니다.


In [7]:
# [2-1] 인덱싱 실행 셀 수정 버전
print("=" * 60)
print("[2-1] 인덱싱 실행")
print("=" * 60)

if parquet_ok:
    cmd = [
        sys.executable, "scripts/index_documents.py",
        "--scenario", "B",
        "--from-parquet", PARQUET_PATH,
        "--collection", COLLECTION,
    ]
else:
    # 옵션 이름을 스크립트 규격에 맞게 수정했습니다.
    cmd = [
        sys.executable, "scripts/index_documents.py",
        "--scenario", "B",
        "--collection", COLLECTION,
        "--step", "all",
        "--documents-dir", "data/documents"  # <-- 여기를 수정했습니다!
    ]

index_ok = run_script(cmd, "인덱싱 스크립트 실행")

[2-1] 인덱싱 실행

▶ 실행: /opt/miniconda3/envs/sprint_env/bin/python3.11 scripts/index_documents.py --scenario B --collection rfp_chunk600 --step all --documents-dir data/documents
------------------------------------------------------------
RFP 문서 인덱싱
  단계      : all
  청킹 전략 : naive | 크기: 1200 | 중첩: 200
  시나리오  : B
  컬렉션    : rfp_chunk600
  Batch API : OFF

[1/2] 문서 로딩 중...
  ✓ 로드 완료: (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp (100835 chars)
  ✓ 로드 완료: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp (49576 chars)
  ✓ 로드 완료: (사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp (71338 chars)
  ✓ 로드 완료: (재)예술경영지원센터_통합 정보시스템 구축 사전 컨설팅.hwp (40929 chars)
  ✓ 로드 완료: 2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp (45538 chars)
  ✓ 로드 완료: BioIN_의��기기산업 종합정보시스템(정보관리기관) 기능개선 사업(2차).hwp (68039 chars)
  ✓ 로드 완료: KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp (124702 chars)
  ✓ 로드 완료: 경기도 안양시_호계체육관 배드민턴장 및 탁구장 예약시스템 구축 용역.hwp (60872 chars)
  ✓ 로드 완료: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp (77392 chars)
  

In [8]:
# 2-2. 산출물 확인
print("=" * 60)
print("[2-2] 인덱싱 산출물 확인")
print("=" * 60)

chunk_file  = Path(f"data/processed/{COLLECTION}_chunks.json")
vectordb_dir = Path("data/vectordb")

chunk_ok = check("청크 JSON 파일 생성", chunk_file.exists(), str(chunk_file))
vdb_ok   = check("vectordb 디렉토리 생성", vectordb_dir.exists(), str(vectordb_dir))

if chunk_ok:
    with open(chunk_file, encoding="utf-8") as f:
        chunks = json.load(f)
    print(f"       총 청크 수: {len(chunks):,}개")
    print(f"       첫 번째 청크 미리보기: {chunks[0]['text'][:80]}...")

if vdb_ok:
    vdb_files = list(vectordb_dir.rglob("*"))
    print(f"       vectordb 파일 수: {len(vdb_files)}개")

[2-2] 인덱싱 산출물 확인
✅ [PASS] 청크 JSON 파일 생성
       data/processed/rfp_chunk600_chunks.json
✅ [PASS] vectordb 디렉토리 생성
       data/vectordb
       총 청크 수: 6,827개
       첫 번째 청크 미리보기: 2024년 ｢벤처확인종합관리시스템 기능 고도화｣ 용역사업
(복수의결권주식, 스톡옵션, 성과조건부주식) -
제안요청서
2024. 03.
목 차
1...
       vectordb 파일 수: 20개


---
## Step 3. RAG 파이프라인 직접 호출
Streamlit 없이 `RAGPipeline`을 직접 임포트해 질문 → 답변 흐름을 테스트합니다.

In [9]:
# 3-1. RAGPipeline 초기화
print("=" * 60)
print("[3-1] RAGPipeline 초기화")
print("=" * 60)

try:
    from configs.config import Config
    from configs import paths
    from src.rag_pipeline import RAGPipeline

    config = Config(
        scenario="B",
        metadata_csv=paths.METADATA_CSV,
        vectordb_dir=paths.VECTORDB_DIR,
        retrieval_method="similarity",
        retrieval_top_k=5,
    )
    pipeline = RAGPipeline(config)
    pipeline.initialize_vectorstore(collection_name=COLLECTION)
    check("RAGPipeline 초기화", True)
except Exception as e:
    check("RAGPipeline 초기화", False, str(e))
    pipeline = None

[3-1] RAGPipeline 초기화
✅ [PASS] RAGPipeline 초기화


In [10]:
# 3-2. 질문 예시로 답변 생성 테스트
print("=" * 60)
print("[3-2] 질문 → 답변 테스트")
print("=" * 60)

TEST_QUESTIONS = [
    "국민연금공단이 발주한 이러닝시스템 사업 요구사항을 정리해 줘.",
    "교육 관련 사업을 발주한 기관들을 모두 알려줘.",
]

if pipeline:
    for i, q in enumerate(TEST_QUESTIONS, 1):
        print(f"\n[Q{i}] {q}")
        try:
            where_filter = pipeline.extract_metadata_filter(q)
            result = pipeline.query(q, retrieval_method="similarity", top_k=5, where=where_filter)

            answer_ok  = bool(result.get("answer"))
            sources_ok = bool(result.get("sources"))
            elapsed    = result.get("elapsed_time", 0)
            check(f"응답 속도 (Q{i})", elapsed < 5, f"{elapsed:.2f}s")

            check(f"답변 생성 (Q{i})", answer_ok)
            check(f"참고문서 반환 (Q{i})", sources_ok,
                  f"검색 문서 {len(result.get('retrieved_docs', []))}개 | 응답 시간: {elapsed:.2f}s")

            if answer_ok:
                print(f"  답변 미리보기: {result['answer'][:120]}...")
            if sources_ok:
                for s in result["sources"][:2]:
                    print(f"  참고: {s.get('발주기관','?')} | {s.get('사업명','?')}")
        except Exception as e:
            check(f"질문 실행 (Q{i})", False, str(e))
    print("\n" + "=" * 60)
    print("[3-3] 할루시네이션 방지 테스트")
    print("=" * 60)
    
    FAKE_QUESTION = "이 문서에 없는 완전히 허구의 사업을 설명해줘"
    
    try:
        fake_result = pipeline.query(FAKE_QUESTION)
        fake_answer = fake_result.get("answer", "")
    
        # 간단 기준: 너무 길고 그럴듯하면 위험
        hallucination_suspect = len(fake_answer) > 100
    
        check("할루시네이션 방지", not hallucination_suspect,
              f"답변 길이: {len(fake_answer)}")
    
        print("  답변:", fake_answer[:120], "...")
    except Exception as e:
        check("할루시네이션 테스트", False, str(e))
else:
    print("⚠️  pipeline이 None — Step 3-1 오류를 먼저 해결하세요.")

[3-2] 질문 → 답변 테스트

[Q1] 국민연금공단이 발주한 이러닝시스템 사업 요구사항을 정리해 줘.
❌ [FAIL] 응답 속도 (Q1)
       31.59s
✅ [PASS] 답변 생성 (Q1)
✅ [PASS] 참고문서 반환 (Q1)
       검색 문서 5개 | 응답 시간: 31.59s
  답변 미리보기: 다음은 국민연금공단이 발주한 2024년 이러닝시스템 운영 용역의 핵심 요구사항을 문서 근거 중심으로 정리한 내용입니다.

1) 기관/사업 정보
- 발주기관: 국민연금공단
- 사업명: 2024년 이러닝시스템 운영 용역...
  참고: 국민연금공단 | 2024년 이러닝시스템 운영 용역
  참고: 국민연금공단 | 사업장 사회보험료 지원 고시 개정에 따른 정보시스템 보완 개발

[Q2] 교육 관련 사업을 발주한 기관들을 모두 알려줘.
❌ [FAIL] 응답 속도 (Q2)
       16.30s
✅ [PASS] 답변 생성 (Q2)
✅ [PASS] 참고문서 반환 (Q2)
       검색 문서 5개 | 응답 시간: 16.30s
  답변 미리보기: 다음 문서들에서 교육 관련 사업을 발주한 기관들입니다.

- 경기도일자리재단
  - 문서: 2025년 통합접수시스템 운영
  - 관련 교육 관련 요구사항: PSR-001 교육지원, PSR-002 기술 이전 및 기술지...
  참고: 재단법인경기도일자리재단 | 2025년 통합접수시스템 운영
  참고: 서울특별시 | 2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용역

[3-3] 할루시네이션 방지 테스트
❌ [FAIL] 할루시네이션 방지
       답변 길이: 436
  답변: 요청하신 “문서에 없는 완전히 허구의 사업”은 본 문서의 근거가 없으므로 제공할 수 없습니다.

대신 아래 대안으로 도와드리겠습니다.
- 현재 문서 내용 요약/정리: 국민연금공단의 2024년 이러닝시스템 운영 용역의 ...


---
## Step 4. RAG 평가
`run_evaluation.py` 를 빠른 테스트(3개 질문)로 실행합니다.

In [11]:
# 4-0. 평가 설정 (필요 시 수정)
EVAL_OUTPUT_DIR = "evaluation"
TEST_LIMIT      = 3      # 빠른 검증용. 전체 실행 시 0으로 변경
EVAL_MODE       = "core" # core / detailed

print(f"평가 설정: mode={EVAL_MODE}, test_limit={TEST_LIMIT}, output={EVAL_OUTPUT_DIR}, collection={COLLECTION}")

평가 설정: mode=core, test_limit=3, output=evaluation, collection=rfp_chunk600


In [12]:
# 4-1. 평가 실행
print("=" * 60)
print("[4-1] run_evaluation.py 실행")
print("=" * 60)

cmd = [
    sys.executable, "scripts/run_evaluation.py",
    "--mode",       EVAL_MODE,
    "--judge",      "off",     # 빠른 테스트: LLM judge 비활성화
    "--bertscore",  "off",     # 빠른 테스트: BERTScore 비활성화
    "--test-limit", str(TEST_LIMIT),
    "--output-dir", EVAL_OUTPUT_DIR,
    "--collection", COLLECTION,
]

eval_ok = run_script(cmd, "평가 스크립트 실행")
if not eval_ok:
    raise RuntimeError("❌ 평가 실패 → 이후 단계 진행 중단")

[4-1] run_evaluation.py 실행

▶ 실행: /opt/miniconda3/envs/sprint_env/bin/python3.11 scripts/run_evaluation.py --mode core --judge off --bertscore off --test-limit 3 --output-dir evaluation --collection rfp_chunk600
------------------------------------------------------------
mode=core | judge=False | bertscore=False | test_limit=3 | collection=rfp_chunk600

config: similarity_k5

[q1] 벤처기업협회가 추진하는 2024년 벤처확인종합관리시스템 기능 고도화에서 DAR-002 데이터 이관의 주요 범위와 사용되는 키는 무엇인가요.
category: single_doc
  elapsed: 10.87s | hit@5: 0.00 | ndcg@5: 0.00
  [follow-up] DAR-008 데이터 암호화 요구사항은 개인정보와 연계 데이터에 대해 무엇을 요구하나요.
    elapsed: 14.87s | hit@5: 1.00

[q2] (사)부산국제영화제가 발주한 2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원 사업의 제안서 평가체계와 우선협상대상자 선정 기준을 정리해 줘.
category: single_doc
  elapsed: 12.51s | hit@5: 1.00 | ndcg@5: 1.00
  [follow-up] 기술능력평가의 정량·정성 부문 배점과 세부 항목 구성을 알려줘.
    elapsed: 56.98s | hit@5: 0.00

[q3] 한국대학스포츠협의회 KUSF 체육특기자 경기기록 관리시스템 개발 사업에서 사업추진계획서 제출 기한은 언제인가?
category: single_doc
  elapsed: 6.22s | hit@5: 1.00 | ndc

In [13]:
# 4-2. 평가 산출물 확인 및 핵심 지표 출력
print("=" * 60)
print("[4-2] 평가 산출물 확인")
print("=" * 60)

eval_dir = Path(EVAL_OUTPUT_DIR)
check("evaluation 디렉토리 생성", eval_dir.exists())

configs_labels = ["similarity_k5", "mmr_k5", "hybrid_k5", "similarity_k10"]

for label in configs_labels:
    csv_file  = eval_dir / f"eval_{label}.csv"
    json_file = eval_dir / f"summary_{label}.json"
    check(f"eval_{label}.csv",     csv_file.exists())
    check(f"summary_{label}.json", json_file.exists())

# 핵심 지표 요약 출력
print("\n핵심 지표 요약")
print(f"{'config':<20} {'hit@5':>8} {'nDCG@5':>8} {'ground':>8} {'p95(s)':>8}")
print("-" * 56)

CORE_METRICS = ["avg_hit_at_5", "avg_ndcg_at_5", "avg_grounded_token_ratio", "p95_elapsed_time"]

for label in configs_labels:
    json_file = eval_dir / f"summary_{label}.json"
    if json_file.exists():
        with open(json_file, encoding="utf-8") as f:
            s = json.load(f)
        hit5   = s.get("avg_hit_at_5", 0)
        ndcg5  = s.get("avg_ndcg_at_5", 0)
        ground = s.get("avg_grounded_token_ratio", 0)
        p95    = s.get("p95_elapsed_time", 0)
        print(f"{label:<20} {hit5:>8.3f} {ndcg5:>8.3f} {ground:>8.3f} {p95:>8.2f}")
        check(f"{label} hit@5 기준", hit5 >= 0.6)
        check(f"{label} ndcg@5 기준", ndcg5 >= 0.5)

[4-2] 평가 산출물 확인
✅ [PASS] evaluation 디렉토리 생성
✅ [PASS] eval_similarity_k5.csv
✅ [PASS] summary_similarity_k5.json
✅ [PASS] eval_mmr_k5.csv
✅ [PASS] summary_mmr_k5.json
✅ [PASS] eval_hybrid_k5.csv
✅ [PASS] summary_hybrid_k5.json
✅ [PASS] eval_similarity_k10.csv
✅ [PASS] summary_similarity_k10.json

핵심 지표 요약
config                  hit@5   nDCG@5   ground   p95(s)
--------------------------------------------------------
similarity_k5           0.667    0.667    0.500    48.59
✅ [PASS] similarity_k5 hit@5 기준
✅ [PASS] similarity_k5 ndcg@5 기준
mmr_k5                  0.833    0.712    0.433    57.70
✅ [PASS] mmr_k5 hit@5 기준
✅ [PASS] mmr_k5 ndcg@5 기준
hybrid_k5               0.667    0.572    0.542    40.54
✅ [PASS] hybrid_k5 hit@5 기준
✅ [PASS] hybrid_k5 ndcg@5 기준
similarity_k10          0.667    0.645    0.591    73.51
✅ [PASS] similarity_k10 hit@5 기준
✅ [PASS] similarity_k10 ndcg@5 기준


---
## Step 5. 릴리즈 게이트
`check_release_gate.py` 를 실행해 최종 PASS / FAIL 를 확인합니다.

In [14]:
# 5-1. 릴리즈 게이트 실행
print("=" * 60)
print("[5-1] check_release_gate.py 실행")
print("=" * 60)

gate_thresholds_file = Path("configs/evaluation/core_gate.default.json")

cmd = [
    sys.executable, "scripts/check_release_gate.py",
    "--output-dir",  EVAL_OUTPUT_DIR,
    "--test-limit",  str(TEST_LIMIT),
    "--judge",       "off",
    "--bertscore",   "off",
    "--allow-fail",  # 노트북 실행이 중단되지 않도록 항상 exit 0
]

# gate threshold 파일이 없으면 --no-run으로 기존 리포트만 읽기
if not gate_thresholds_file.exists():
    print(f"⚠️  {gate_thresholds_file} 없음 → 기존 gate report만 읽습니다.")
    cmd.append("--no-run")
else:
    cmd.extend(["--gate-thresholds", str(gate_thresholds_file)])

gate_ok = run_script(cmd, "릴리즈 게이트 실행")

[5-1] check_release_gate.py 실행

▶ 실행: /opt/miniconda3/envs/sprint_env/bin/python3.11 scripts/check_release_gate.py --output-dir evaluation --test-limit 3 --judge off --bertscore off --allow-fail --gate-thresholds configs/evaluation/core_gate.default.json
------------------------------------------------------------
mode=core | judge=False | bertscore=False | test_limit=3 | collection=rfp_chunk600

config: similarity_k5

[q1] 벤처기업협회가 추진하는 2024년 벤처확인종합관리시스템 기능 고도화에서 DAR-002 데이터 이관의 주요 범위와 사용되는 키는 무엇인가요.
category: single_doc
  elapsed: 15.67s | hit@5: 0.00 | ndcg@5: 0.00
  [follow-up] DAR-008 데이터 암호화 요구사항은 개인정보와 연계 데이터에 대해 무엇을 요구하나요.
    elapsed: 19.04s | hit@5: 1.00

[q2] (사)부산국제영화제가 발주한 2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원 사업의 제안서 평가체계와 우선협상대상자 선정 기준을 정리해 줘.
category: single_doc
  elapsed: 15.88s | hit@5: 1.00 | ndcg@5: 1.00
  [follow-up] 기술능력평가의 정량·정성 부문 배점과 세부 항목 구성을 알려줘.
    elapsed: 90.75s | hit@5: 0.00

[q3] 한국대학스포츠협의회 KUSF 체육특기자 경기기록 관리시스템 개발 사업에서 사업추진계획서 제출 기한은 언제인가?
category: sing

In [15]:
# 5-2. gate_report_core.json 파싱 및 최종 결과 출력
print("=" * 60)
print("[5-2] 릴리즈 게이트 최종 결과")
print("=" * 60)

gate_report_path = Path(EVAL_OUTPUT_DIR) / "gate_report_core.json"

if gate_report_path.exists():
    with open(gate_report_path, encoding="utf-8") as f:
        gate_report = json.load(f)

    print(f"best_config: {gate_report.get('best_config')}\n")
    print(f"{'config':<20} {'gate':>6} {'pass':>10} {'ratio':>8}")
    print("-" * 50)

    any_passed = False
    for label, row in gate_report.get("configs", {}).items():
        gp       = row.get("gate_passed", False)
        pc       = f"{row.get('pass_count',0)}/{row.get('total_count',0)}"
        ratio    = row.get("pass_ratio", 0.0)
        gate_str = "PASS" if gp else "FAIL"
        icon     = "✅" if gp else "❌"
        print(f"{icon} {label:<18} {gate_str:>6} {pc:>10} {ratio:>8.2f}")
        if gp:
            any_passed = True

    print()
    if any_passed:
        print("🎉 RESULT: PASS — 하나 이상의 config가 릴리즈 기준을 통과했습니다.")
    else:
        print("⚠️  RESULT: FAIL — 릴리즈 기준을 통과한 config가 없습니다.")
        print("   Step 4 지표를 확인하고 threshold 또는 retrieval 설정을 조정하세요.")
else:
    check("gate_report_core.json 생성", False, "Step 5-1 오류를 먼저 확인하세요.")

[5-2] 릴리즈 게이트 최종 결과
best_config: mmr_k5

config                 gate       pass    ratio
--------------------------------------------------
❌ similarity_k5        FAIL        1/5     0.20
❌ mmr_k5               FAIL        2/5     0.40
❌ hybrid_k5            FAIL        1/5     0.20
❌ similarity_k10       FAIL        1/5     0.20

⚠️  RESULT: FAIL — 릴리즈 기준을 통과한 config가 없습니다.
   Step 4 지표를 확인하고 threshold 또는 retrieval 설정을 조정하세요.


---
## 최종 요약

In [16]:
# 전체 체크 결과 요약
print("=" * 60)
print("전체 파이프라인 체크 요약")
print("=" * 60)

summary_checks = [
    ("1. 환경 점검",     "패키지 / API 키 / 파일 구조 / 모델명"),
    ("2. 인덱싱",        f"청크 JSON + vectordb 생성 ({COLLECTION})"),
    ("3. RAG 파이프라인","질문 → 답변 + 참고 문서 반환"),
    ("4. 평가",          f"4가지 config 평가 결과 CSV/JSON 생성"),
    ("5. 릴리즈 게이트", "gate_report_core.json PASS/FAIL"),
]

for step, desc in summary_checks:
    print(f"  {step:<20} {desc}")

print()
print("📌 각 단계 셀의 ✅/❌ 출력을 확인하세요.")
print("   ❌가 있는 단계는 해당 섹션의 오류 메시지를 참고해 수정 후 재실행하세요.")

전체 파이프라인 체크 요약
  1. 환경 점검             패키지 / API 키 / 파일 구조 / 모델명
  2. 인덱싱               청크 JSON + vectordb 생성 (rfp_chunk600)
  3. RAG 파이프라인         질문 → 답변 + 참고 문서 반환
  4. 평가                4가지 config 평가 결과 CSV/JSON 생성
  5. 릴리즈 게이트           gate_report_core.json PASS/FAIL

📌 각 단계 셀의 ✅/❌ 출력을 확인하세요.
   ❌가 있는 단계는 해당 섹션의 오류 메시지를 참고해 수정 후 재실행하세요.
